# Music Recommender System — Collaborative Filtering

Collaborative filtering recommends music based on user listening patterns:
- **User-based**: 'Users who liked X also liked Y'
- **Item-based**: 'Songs similar to X (based on who listens to them)'

No audio analysis needed — just user-item interaction data.

In [ ]:
!pip install numpy scipy matplotlib pandas scikit-learn

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
n_users, n_songs = 50, 30

# Genre preferences (3 genres)
user_genres = np.random.choice(3, n_users)
song_genres = np.random.choice(3, n_songs)

# Generate ratings: higher if user and song share genre
ratings = np.zeros((n_users, n_songs))
for u in range(n_users):
    for s in range(n_songs):
        if user_genres[u] == song_genres[s]:
            ratings[u, s] = np.random.choice([4, 5], p=[0.3, 0.7])
        else:
            ratings[u, s] = np.random.choice([0, 1, 2, 3], p=[0.4, 0.2, 0.2, 0.2])

# Make sparse (not everyone rates everything)
mask = np.random.random((n_users, n_songs)) > 0.6
ratings *= mask

song_names = [f"Song_{i}" for i in range(n_songs)]
user_names = [f"User_{i}" for i in range(n_users)]

df = pd.DataFrame(ratings, index=user_names, columns=song_names)
print(f"Rating matrix: {n_users} users x {n_songs} songs")
print(f"Sparsity: {(ratings == 0).sum() / ratings.size:.1%}")
df.head()

## User-Based Collaborative Filtering

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute user-user similarity
user_sim = cosine_similarity(ratings)
np.fill_diagonal(user_sim, 0)  # Don't recommend based on self

def recommend_user_based(user_idx, ratings, user_sim, top_k=5, n_neighbors=10):
    """Recommend songs for a user based on similar users' ratings."""
    # Find most similar users
    similar_users = np.argsort(user_sim[user_idx])[::-1][:n_neighbors]
    sim_weights = user_sim[user_idx][similar_users]
    
    # Weighted average of similar users' ratings
    weighted_ratings = np.zeros(ratings.shape[1])
    for neighbor, weight in zip(similar_users, sim_weights):
        weighted_ratings += weight * ratings[neighbor]
    
    if sim_weights.sum() > 0:
        weighted_ratings /= sim_weights.sum()
    
    # Exclude already-rated songs
    already_rated = ratings[user_idx] > 0
    weighted_ratings[already_rated] = -1
    
    # Top-k recommendations
    top_songs = np.argsort(weighted_ratings)[::-1][:top_k]
    return [(song_names[s], weighted_ratings[s]) for s in top_songs]

# Recommend for User_0
target_user = 0
recs = recommend_user_based(target_user, ratings, user_sim)
print(f"Recommendations for {user_names[target_user]}:")
print(f"  (Genre preference: {user_genres[target_user]})\n")
for song, score in recs:
    idx = song_names.index(song)
    print(f"  {song} (predicted: {score:.2f}, genre: {song_genres[idx]})")

## Item-Based Collaborative Filtering

In [ ]:
import matplotlib.pyplot as plt

# Compute song-song similarity (transpose ratings so songs are rows)
song_sim = cosine_similarity(ratings.T)
np.fill_diagonal(song_sim, 0)

# Show top-5 similar songs for a few examples
for song_idx in [0, 5, 15]:
    similar = np.argsort(song_sim[song_idx])[::-1][:5]
    print(f"\nSongs similar to {song_names[song_idx]} (genre {song_genres[song_idx]}):")
    for s in similar:
        print(f"  {song_names[s]} (sim: {song_sim[song_idx, s]:.3f}, genre: {song_genres[s]})")

# Visualize song similarity matrix
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(song_sim, cmap='YlOrRd')
ax.set_title('Song-Song Similarity Matrix (Item-Based CF)')
ax.set_xlabel('Songs')
ax.set_ylabel('Songs')
plt.colorbar(im, label='Cosine Similarity')
plt.tight_layout()
plt.show()

## Matrix Factorization (SVD)

In [ ]:
from numpy.linalg import svd

# SVD decomposition
U, sigma, Vt = svd(ratings, full_matrices=False)

# Keep top-k latent factors
k = 5
U_k = U[:, :k]
sigma_k = np.diag(sigma[:k])
Vt_k = Vt[:k, :]

# Reconstruct rating matrix
ratings_pred = U_k @ sigma_k @ Vt_k

# Visualize latent factors
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# User latent factors (first 2 dimensions)
scatter = axes[0].scatter(U_k[:, 0], U_k[:, 1], c=user_genres, cmap='Set1', s=60, alpha=0.7)
axes[0].set_title('Users in Latent Space (colored by genre preference)')
axes[0].set_xlabel('Factor 1')
axes[0].set_ylabel('Factor 2')
axes[0].legend(*scatter.legend_elements(), title='Genre')

# Song latent factors
scatter = axes[1].scatter(Vt_k[0, :], Vt_k[1, :], c=song_genres, cmap='Set1', s=60, alpha=0.7)
axes[1].set_title('Songs in Latent Space (colored by true genre)')
axes[1].set_xlabel('Factor 1')
axes[1].set_ylabel('Factor 2')
axes[1].legend(*scatter.legend_elements(), title='Genre')

plt.tight_layout()
plt.show()

# Recommend using SVD predictions
target_user = 0
predicted = ratings_pred[target_user]
already_rated = ratings[target_user] > 0
predicted[already_rated] = -1
top_songs = np.argsort(predicted)[::-1][:5]

print(f"\nSVD Recommendations for {user_names[target_user]}:")
for s in top_songs:
    print(f"  {song_names[s]} (predicted: {predicted[s]:.2f}, genre: {song_genres[s]})")

## Evaluation

How do we know if recommendations are good?

In [ ]:
from sklearn.model_selection import train_test_split

# Get all non-zero ratings as (user, song, rating) triples
nonzero = np.argwhere(ratings > 0)
rating_values = ratings[ratings > 0]

# Train/test split
train_idx, test_idx = train_test_split(range(len(nonzero)), test_size=0.2, random_state=42)

# Build train matrix
train_ratings = np.zeros_like(ratings)
for idx in train_idx:
    u, s = nonzero[idx]
    train_ratings[u, s] = rating_values[idx]

# SVD on train data
U_tr, sigma_tr, Vt_tr = svd(train_ratings, full_matrices=False)
pred_tr = U_tr[:, :k] @ np.diag(sigma_tr[:k]) @ Vt_tr[:k, :]

# RMSE on test set
test_true = []
test_pred = []
for idx in test_idx:
    u, s = nonzero[idx]
    test_true.append(rating_values[idx])
    test_pred.append(pred_tr[u, s])

rmse = np.sqrt(np.mean((np.array(test_true) - np.array(test_pred))**2))
print(f"RMSE on test set: {rmse:.3f}")
print(f"(Baseline — always predict mean: {np.std(test_true):.3f})")

# Precision@K: Of top-K recommended, how many are actually liked (rating >= 4)?
K = 5
precisions = []
for u in range(n_users):
    # Get songs the user actually liked in test
    test_liked = set()
    for idx in test_idx:
        uu, ss = nonzero[idx]
        if uu == u and rating_values[idx] >= 4:
            test_liked.add(ss)
    if len(test_liked) == 0:
        continue
    
    # Top-K predictions (exclude training items)
    scores = pred_tr[u].copy()
    scores[train_ratings[u] > 0] = -1
    top_k_songs = set(np.argsort(scores)[::-1][:K])
    
    precision = len(top_k_songs & test_liked) / K
    precisions.append(precision)

print(f"Precision@{K}: {np.mean(precisions):.3f}")

## Limitations and Biases

- **Popularity bias**: Popular songs get recommended more
- **Filter bubbles**: Users get trapped in narrow genres
- **Cold start**: New songs/users have no data
- **Cultural bias**: Training data reflects existing listening patterns